In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import idx2numpy
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from mlp_model.losses.bce_loss import BCELoss
from mlp_model.losses.cce_loss import CCELoss
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.activations.relu import ReLU
from mlp_model.activations.tanh import Tanh
from mlp_model.activations.sigmoid import Sigmoid
import visualisation.classification_visualise as cl_vs
import classification_training_methods as cl_train

In [ ]:
train_images_filepath = Path(project_root) / "data/mnist/train" / "train-images-idx3-ubyte"
train_labels_filepath = Path(project_root) / "data/mnist/train" / "train-labels-idx1-ubyte"

In [ ]:
X = idx2numpy.convert_from_file(str(train_images_filepath))
y = idx2numpy.convert_from_file(str(train_labels_filepath))

In [ ]:
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=600, random_state=42, stratify=y)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample)

In [ ]:
X_train = X_train.reshape(X_train.shape[0], -1).astype(np.float32) / 255.0
X_val = X_val.reshape(X_val.shape[0], -1).astype(np.float32) / 255.0

## Ocena różnych architektur

In [ ]:
archis = {
    "64": (64,),
    "128": (128,),
    "256": (256,),
    "512": (512,),

    "64-64": (64, 64),
    "128-64": (128, 64),
    "128-128": (128, 128),
    "256-128": (256, 128),
    "256-256": (256, 256),
    "512-256": (512, 256),
    "512-512": (512, 512),

    "128-64-32": (128, 64, 32),
    "256-128-64": (256, 128, 64),
    "256-256-128": (256, 256, 128),
    "512-256-128": (512, 256, 128),
    "512-512-256": (512, 512, 256),

    "128-128-64-32": (128, 128, 64, 32),
    "256-128-64-32": (256, 128, 64, 32),
    "256-256-128-64": (256, 256, 128, 64),
    "512-256-128-64": (512, 256, 128, 64),
    "512-512-256-128": (512, 512, 256, 128),
    "1024-512-256-128": (1024, 512, 256, 128),
}

trainings = {}

for name, hidden_sizes in archis.items():
    trainings[name] = cl_train.run_training(hidden_sizes=hidden_sizes, X_train=X_train, y_train=y_train, X_val=X_val,
                                            y_val=y_val, activation_hidden=ReLU(), epochs=30,
                                            loss_cls=CCELoss(), learning_rate=0.01, classes_count=10)

cl_vs.plot_loss(trainings, "train_loss")
cl_vs.plot_loss(trainings, "val_loss")

display(cl_train.summarize_results(trainings))